# DatasetMain Commitment Juncture Threshold Sweep

This notebook examines commitment junctures under multiple threshold values `tau`.

Definitions used here:

- A commitment juncture is a change in **consecutive counterfactual deception rate** between adjacent saved localization sentences.
- "Consecutive sentences" means consecutive entries in each saved localization trace after sorting by the saved sentence index.
- Positive commitment: `Delta_k = p_k - p_{k-1} > tau`
- Negative commitment: `Delta_k = p_k - p_{k-1} < -tau`
- We require `num_valid > 10` on **both** sides of the pair before counting a juncture.
- Coverage is the share of examples with at least one qualifying juncture among examples that have at least one valid consecutive sentence pair.

The notebook reports:

1. Overall threshold-sensitivity tables for positive and negative junctures.
2. Histograms for `tau in {0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7}`.
3. Per-model summaries pooled across environments.
4. Per-model-by-environment summaries.

By default the notebook reads the raw saved localization JSON files, for example `DatasetMain/bs/DeepSeek-R1-Distill-Llama-8B/localization/sentence_localization_2026-03-11_13-53-37_game_0_turn_0_state_0_sample_8.json`. If you explicitly want the cached prefix parquet path instead, set `PREFERRED_SOURCE = "parquet"`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import datasetmain_commitment_juncture_prevalence_lib as cj
import datasetmain_commitment_juncture_threshold_lib as cjt


cj = importlib.reload(cj)
cjt = importlib.reload(cjt)

DATASETMAIN_ROOT = cjt.DATASETMAIN_ROOT
TAU_VALUES = cjt.TAU_VALUES
DEFAULT_TAU = 0.3
MIN_VALID = cjt.DEFAULT_MIN_VALID
PREFERRED_SOURCE = cjt.DEFAULT_SOURCE_KIND
EXPECTED_JSONS_PER_BUNDLE = 5000
MIN_EXPECTED_JSONS_PER_BUNDLE = 4900
INCLUDE_SENTENCE_TEXT = False
MAX_JSON_FILES_PER_BUNDLE = None
SHOW_PROGRESS = True
PROGRESS_LEVEL = 'bundle'
SAVE_ARTIFACTS = False
ARTIFACT_DIR = Path('/playpen-ssd/smerrill/deception2/Notebooks/commitment_juncture_threshold_outputs')

plt.style.use('seaborn-v0_8-whitegrid')
pd.options.display.max_columns = 200
pd.options.display.max_colwidth = 220


def md(text: str) -> None:
    display(Markdown(text))


def ensure_artifact_dir() -> Path:
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    return ARTIFACT_DIR


def maybe_save_table(df: pd.DataFrame, stem: str) -> None:
    if not SAVE_ARTIFACTS:
        return
    out_dir = ensure_artifact_dir()
    df.to_csv(out_dir / f'{stem}.csv', index=False)


def display_threshold_table(df: pd.DataFrame) -> None:
    format_map = {}
    for column in ['Threshold']:
        if column in df.columns:
            format_map[column] = '{:.1f}'
    for column in ['Coverage']:
        if column in df.columns:
            format_map[column] = '{:.1%}'
    for column in ['Mean Delta', 'Mean Δ_k', 'Pre-rate', 'Post-rate']:
        if column in df.columns:
            format_map[column] = '{:.3f}'
    for column in ['Examples', 'Pairs']:
        if column in df.columns:
            format_map[column] = '{:.0f}'
    display(df.style.hide(axis='index').format(format_map, na_rep=''))


def make_table(
    summary_df: pd.DataFrame,
    *,
    include_group_columns: bool = True,
    include_counts: bool = True,
) -> pd.DataFrame:
    return cjt.format_threshold_summary_table(
        summary_df,
        include_group_columns=include_group_columns,
        include_counts=include_counts,
    ).rename(columns={'Mean Delta': 'Mean Δ_k'})


In [ ]:
inventory_df, prefix_df, parse_error_df = cjt.load_datasetmain_threshold_prefix_df(
    DATASETMAIN_ROOT,
    include_sentence_text=INCLUDE_SENTENCE_TEXT,
    max_json_files_per_bundle=MAX_JSON_FILES_PER_BUNDLE,
    preferred_source=PREFERRED_SOURCE,
    show_progress=SHOW_PROGRESS,
    progress_level=PROGRESS_LEVEL,
)
pair_df = cjt.build_consecutive_pair_df(prefix_df, min_valid=MIN_VALID)
valid_pair_df = pair_df.loc[pair_df['pair_is_valid'].fillna(False)].copy()

if not inventory_df['source_kind'].astype(str).eq('localization_json').all():
    raise AssertionError('Expected every bundle to be loaded from raw localization JSON files.')

inventory_table_df = inventory_df.loc[
    :,
    [
        'model_display',
        'env_display',
        'source_kind',
        'json_file_count',
        'loaded_examples',
        'loaded_rows',
    ],
].rename(
    columns={
        'model_display': 'Model',
        'env_display': 'Environment',
        'source_kind': 'Loaded From',
        'json_file_count': 'Localization JSONs',
        'loaded_examples': 'Examples',
        'loaded_rows': 'Prefix Rows',
    }
)
inventory_table_df['Gap vs 5000'] = inventory_table_df['Localization JSONs'] - EXPECTED_JSONS_PER_BUNDLE
inventory_table_df['Near 5k'] = inventory_table_df['Localization JSONs'].between(MIN_EXPECTED_JSONS_PER_BUNDLE, EXPECTED_JSONS_PER_BUNDLE)

md('## Load Summary')
display(inventory_table_df.style.hide(axis='index'))

valid_example_count = int(valid_pair_df.loc[:, cjt.EXAMPLE_KEY_COLUMNS].drop_duplicates().shape[0]) if not valid_pair_df.empty else 0
json_count_min = int(inventory_df['json_file_count'].min()) if not inventory_df.empty else 0
json_count_max = int(inventory_df['json_file_count'].max()) if not inventory_df.empty else 0
md(
    f'Read directly from raw localization JSON files. Preferred source: `{PREFERRED_SOURCE}`. '
    f'Per-bundle JSON counts range from `{json_count_min:,}` to `{json_count_max:,}` with an expected target of about `{EXPECTED_JSONS_PER_BUNDLE:,}` per model x environment. '
    f'Loaded `{len(prefix_df):,}` prefix rows and `{len(pair_df):,}` consecutive sentence pairs. '
    f'Valid pairs after requiring `num_valid > {MIN_VALID}` on both sides: `{len(valid_pair_df):,}` across `{valid_example_count:,}` examples. '
    f'Bundle parse warnings/errors captured: `{len(parse_error_df):,}`. '
    f'Progress settings: `SHOW_PROGRESS={SHOW_PROGRESS}`, `PROGRESS_LEVEL={PROGRESS_LEVEL}`.'
)

non_exact_json_count_df = inventory_table_df.loc[
    ~inventory_table_df['Localization JSONs'].eq(EXPECTED_JSONS_PER_BUNDLE),
    ['Model', 'Environment', 'Localization JSONs', 'Gap vs 5000'],
].reset_index(drop=True)
if not non_exact_json_count_df.empty:
    md('### Bundles Not Exactly 5000 JSONs')
    display(non_exact_json_count_df.style.hide(axis='index'))
    maybe_save_table(non_exact_json_count_df, 'non_exact_json_counts')

if not parse_error_df.empty:
    parse_error_table_df = parse_error_df.loc[:, [column for column in ['bundle_dir', 'path', 'source_kind', 'error'] if column in parse_error_df.columns]].drop_duplicates()
    md('### Parse Warnings')
    display(parse_error_table_df.style.hide(axis='index'))
    maybe_save_table(parse_error_table_df, 'parse_warnings')

maybe_save_table(inventory_table_df, 'inventory')


In [ ]:
positive_overall_summary_df = cjt.summarize_threshold_sweep(
    pair_df,
    tau_values=TAU_VALUES,
    polarity='positive',
)
negative_overall_summary_df = cjt.summarize_threshold_sweep(
    pair_df,
    tau_values=TAU_VALUES,
    polarity='negative',
)

positive_model_summary_df = cjt.summarize_threshold_sweep(
    pair_df,
    tau_values=TAU_VALUES,
    polarity='positive',
    groupby_columns=['model_display'],
)
negative_model_summary_df = cjt.summarize_threshold_sweep(
    pair_df,
    tau_values=TAU_VALUES,
    polarity='negative',
    groupby_columns=['model_display'],
)

positive_env_model_summary_df = cjt.summarize_threshold_sweep(
    pair_df,
    tau_values=TAU_VALUES,
    polarity='positive',
    groupby_columns=['model_display', 'env_display'],
)
negative_env_model_summary_df = cjt.summarize_threshold_sweep(
    pair_df,
    tau_values=TAU_VALUES,
    polarity='negative',
    groupby_columns=['model_display', 'env_display'],
)

positive_overall_table_df = make_table(positive_overall_summary_df, include_group_columns=False, include_counts=True)
negative_overall_table_df = make_table(negative_overall_summary_df, include_group_columns=False, include_counts=True)

positive_model_table_df = make_table(positive_model_summary_df, include_group_columns=True, include_counts=True)
negative_model_table_df = make_table(negative_model_summary_df, include_group_columns=True, include_counts=True)

positive_env_model_table_df = make_table(positive_env_model_summary_df, include_group_columns=True, include_counts=True)
negative_env_model_table_df = make_table(negative_env_model_summary_df, include_group_columns=True, include_counts=True)

positive_model_focus_table_df = positive_model_table_df.loc[positive_model_table_df['Threshold'].eq(DEFAULT_TAU)].reset_index(drop=True)
negative_model_focus_table_df = negative_model_table_df.loc[negative_model_table_df['Threshold'].eq(DEFAULT_TAU)].reset_index(drop=True)

positive_env_model_focus_table_df = positive_env_model_table_df.loc[positive_env_model_table_df['Threshold'].eq(DEFAULT_TAU)].reset_index(drop=True)
negative_env_model_focus_table_df = negative_env_model_table_df.loc[negative_env_model_table_df['Threshold'].eq(DEFAULT_TAU)].reset_index(drop=True)

maybe_save_table(positive_overall_summary_df, 'positive_overall_summary_raw')
maybe_save_table(negative_overall_summary_df, 'negative_overall_summary_raw')
maybe_save_table(positive_model_summary_df, 'positive_model_summary_raw')
maybe_save_table(negative_model_summary_df, 'negative_model_summary_raw')
maybe_save_table(positive_env_model_summary_df, 'positive_env_model_summary_raw')
maybe_save_table(negative_env_model_summary_df, 'negative_env_model_summary_raw')


## Overall Threshold Tables

The first table matches the paper-style threshold sweep for positive commitments. The second table gives the same summary for negative commitments separately.


In [ ]:
md('### Positive Commitments: Toward Deception')
display_threshold_table(positive_overall_table_df)

md('### Negative Commitments: Toward Truthfulness')
display_threshold_table(negative_overall_table_df)

maybe_save_table(positive_overall_table_df, 'positive_overall_table')
maybe_save_table(negative_overall_table_df, 'negative_overall_table')


In [ ]:
def plot_threshold_histograms(
    pair_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    *,
    tau_values: tuple[float, ...] | list[float],
    polarity: str,
    color: str,
) -> plt.Figure:
    tau_list = [float(value) for value in tau_values]
    n_cols = 4
    n_rows = int(np.ceil(len(tau_list) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.6 * n_rows), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).reshape(n_rows, n_cols)
    bins = np.linspace(0.0, 1.0, 31) if polarity == 'positive' else np.linspace(-1.0, 0.0, 31)
    summary_lookup = summary_df.set_index('tau')

    for ax, tau in zip(axes.ravel(), tau_list):
        event_df = cjt.select_threshold_events(pair_df, tau=tau, polarity=polarity)
        coverage = float(summary_lookup.loc[tau, 'coverage']) if tau in summary_lookup.index else float('nan')
        ax.hist(event_df['delta_deception_rate'], bins=bins, color=color, edgecolor='white', alpha=0.90)
        ax.axvline(tau if polarity == 'positive' else -tau, color='#222222', linestyle='--', linewidth=1.0)
        ax.set_title(f'tau={tau:.1f}\nn={len(event_df):,}, coverage={coverage:.1%}')
        ax.set_xlabel('Delta_k')
        ax.set_ylabel('Pairs')
        if polarity == 'positive':
            ax.set_xlim(0.0, 1.0)
        else:
            ax.set_xlim(-1.0, 0.0)

    for ax in axes.ravel()[len(tau_list):]:
        ax.axis('off')

    direction = 'positive' if polarity == 'positive' else 'negative'
    fig.suptitle(f'Commitment-juncture delta histograms ({direction})', fontsize=16, y=1.02)
    fig.tight_layout()
    return fig


positive_hist_fig = plot_threshold_histograms(
    pair_df,
    positive_overall_summary_df,
    tau_values=TAU_VALUES,
    polarity='positive',
    color='#d96b3b',
)
negative_hist_fig = plot_threshold_histograms(
    pair_df,
    negative_overall_summary_df,
    tau_values=TAU_VALUES,
    polarity='negative',
    color='#3f6b9a',
)
plt.show()


## Per-Model Numbers

The focus tables below use `tau = 0.3`. The full long-format per-model tables for all thresholds remain available in `positive_model_table_df` and `negative_model_table_df`.


In [ ]:
md(f'### Positive Commitments by Model at tau={DEFAULT_TAU:.1f}')
display_threshold_table(positive_model_focus_table_df)

md(f'### Negative Commitments by Model at tau={DEFAULT_TAU:.1f}')
display_threshold_table(negative_model_focus_table_df)

maybe_save_table(positive_model_table_df, 'positive_model_table_all_tau')
maybe_save_table(negative_model_table_df, 'negative_model_table_all_tau')
maybe_save_table(positive_model_focus_table_df, 'positive_model_table_tau_0p3')
maybe_save_table(negative_model_focus_table_df, 'negative_model_table_tau_0p3')


## Per Model x Environment

These tables also focus on `tau = 0.3` for readability. The complete long-format threshold sweeps remain available in `positive_env_model_table_df` and `negative_env_model_table_df`.


In [ ]:
md(f'### Positive Commitments by Model x Environment at tau={DEFAULT_TAU:.1f}')
display_threshold_table(positive_env_model_focus_table_df)

md(f'### Negative Commitments by Model x Environment at tau={DEFAULT_TAU:.1f}')
display_threshold_table(negative_env_model_focus_table_df)

maybe_save_table(positive_env_model_table_df, 'positive_env_model_table_all_tau')
maybe_save_table(negative_env_model_table_df, 'negative_env_model_table_all_tau')
maybe_save_table(positive_env_model_focus_table_df, 'positive_env_model_table_tau_0p3')
maybe_save_table(negative_env_model_focus_table_df, 'negative_env_model_table_tau_0p3')


## Handy Objects

Useful data frames left in memory after running the notebook:

- `inventory_df`: bundle-level load summary
- `prefix_df`: per-sentence counterfactual rows
- `pair_df`: consecutive-sentence pairs with `Delta_k`
- `positive_overall_summary_df`, `negative_overall_summary_df`
- `positive_model_summary_df`, `negative_model_summary_df`
- `positive_env_model_summary_df`, `negative_env_model_summary_df`

If you want different views later, the core helper to reuse is:

```python
cjt.select_threshold_events(pair_df, tau=0.3, polarity='positive')
```
